# Databricks / PySpark dla Data Analyst/Engineer — Moduł 14: Praca w Databricks Workspace

Przez trzynaście modułów pisaliśmy kod PySparka, który działa identycznie lokalnie i w
Databricks. Ten moduł jest inny — nie uczy nowej składni Sparka, tylko samej **platformy
Databricks**: jak wygląda workspace, jak uruchamiać notebooki na prawdziwym klastrze,
jak parametryzować i planować zadania. To wiedza praktyczna, potrzebna, żeby przejść od
"umiem pisać PySparka" do "umiem pracować w zespole danych na Databricks".

## Spis treści
1. [Anatomia Databricks Workspace](#sec1)
2. [Notebooki w Databricks](#sec2)
3. [Klastry i Compute](#sec3)
4. [`dbutils` — scyzoryk szwajcarski notebooków](#sec4)
5. [Widgety — parametryzacja notebooków](#sec5)
6. [Jobs i Workflows — planowanie zadań](#sec6)
7. [Praktyczny spacerek po Databricks Free Edition](#sec7)
8. [Podsumowanie i ćwiczenia](#sec8)


> ⚠️ **Ten moduł jest inny niż poprzednie**
>
> Kod w Modułach 1-13 uruchamialiśmy lokalnie, bo pisaliśmy "czystego" PySparka. Ten moduł opisuje elementy platformy Databricks (`dbutils`, widgety, Jobs UI), które istnieją WYŁĄCZNIE wewnątrz prawdziwego środowiska Databricks -- nie da się ich uruchomić lokalnie ani przetestować w tym notebooku. Dlatego przykłady kodu w tym module pokazujemy jako fragmenty do przeczytania i wypróbowania samodzielnie w Databricks Free Edition (poznanym w Module 1), a nie jako uruchamialne komórki.

<a id="sec1"></a>
## 1. Anatomia Databricks Workspace

Po zalogowaniu się do Databricks (Free Edition albo płatnej wersji firmowej) trafiasz do
**Workspace** -- głównego środowiska pracy. Pasek boczny zawiera kilka kluczowych sekcji:

- **Workspace** -- drzewo folderów z Twoimi notebookami, zapytaniami SQL i plikami;
  odpowiednik systemu plików, ale dla zasobów Databricks. Możesz tu tworzyć foldery,
  udostępniać notebooki współpracownikom i przeglądać historię zmian.
- **Repos** (albo **Git folders**) -- integracja z Git (GitHub, GitLab, Azure DevOps):
  klonujesz repozytorium bezpośrednio do workspace'u, żeby notebooki żyły w tym samym
  repo co reszta kodu zespołu, z pełnym `git pull`/`push`/`commit` z poziomu UI.
- **Catalog** (Catalog Explorer / Unity Catalog) -- przeglądarka wszystkich katalogów,
  schematów i tabel, które poznaliśmy koncepcyjnie w Module 11. Stąd sprawdzisz schemat
  tabeli, jej uprawnienia i pochodzenie (lineage) bez pisania ani linijki kodu.
- **Compute** -- lista klastrów i SQL Warehouse'ów: tu je tworzysz, uruchamiasz,
  zatrzymujesz i monitorujesz zużycie zasobów.
- **Workflows** (dawniej Jobs) -- planowanie i orkiestracja zadań: tu definiujesz, KIEDY
  i W JAKIEJ KOLEJNOŚCI mają się uruchamiać Twoje notebooki i skrypty (sekcja 6).

Wszystkie te sekcje są ze sobą powiązane: notebook z Workspace uruchamiasz na klastrze z
Compute, czytający tabele z Catalog, a całość planujesz jako zadanie w Workflows.

> 🧱 **Free Edition a wersja firmowa**
>
> Databricks Free Edition ma uproszczoną nawigację (mniej ustawień administracyjnych), ale wszystkie powyższe sekcje są obecne i działają tak samo koncepcyjnie -- to, czego nauczysz się tutaj, przenosi się wprost na płatne środowisko firmowe.

<a id="sec2"></a>
## 2. Notebooki w Databricks

Notebooki Databricks przypominają Jupytera (na którym pracowaliśmy cały kurs), ale mają
kilka funkcji, których Jupyter nie ma:

**Komórki wielojęzykowe.** Domyślny język notebooka ustawiasz raz, ale KAŻDA komórka może
używać innego języka dzięki tzw. *magic commands* na początku komórki:

```
%python
# kod PySparka (domyślny język tego kursu)
df = spark.table("klienci")

%sql
-- czysty SQL, dokładnie jak w kursie SQL/PostgreSQL
SELECT segment, COUNT(*) FROM klienci GROUP BY segment

%scala
// kod Scali -- ten sam SparkSession, inny język
val df = spark.table("klienci")

%md
### To jest komórka Markdown, taka jak w Jupyterze
```

Dzięki temu jeden notebook może np. wczytać dane w PySparku, przeanalizować je zapytaniem
SQL i podsumować wynik w komórce Markdown -- bez przełączania się między plikami.

**Historia wersji i komentarze.** Databricks automatycznie zapisuje historię zmian
notebooka (podobnie do commitów w Git) -- możesz wrócić do dowolnej wcześniejszej wersji.
Możesz też zostawiać komentarze przy konkretnych komórkach, co ułatwia code review w
zespole (podobnie jak komentarze w Pull Requestach na GitHubie).

**`%run`.** Magic command `%run ./inny_notebook` uruchamia inny notebook "w miejscu" --
wszystkie jego zmienne i funkcje stają się dostępne w bieżącym notebooku. To prosty sposób
na współdzielenie kodu (np. wspólnych funkcji pomocniczych) między notebookami bez
budowania osobnej paczki Pythona.

> ⚡ **Ten kurs w Databricks**
>
> Wszystkie nasze notebooki z Modułów 1-13 są napisane w czystym PySparku (komórki bez żadnego `%magic` na początku = domyślny Python), więc zaimportowane do Databricks będą działać bez zmian -- Databricks po prostu potraktuje każdą komórkę jako `%python`.

<a id="sec3"></a>
## 3. Klastry i Compute

Zanim uruchomisz jakikolwiek kod, notebook musi być "podłączony" (*attached*) do
działającego **klastra** (compute) -- to on faktycznie wykonuje obliczenia, które w tym
kursie robiliśmy na jednej, lokalnej maszynie.

**Typy compute:**

- **All-purpose cluster** -- klaster do interaktywnej pracy w notebookach: uruchamiasz go
  ręcznie, pracujesz, a on stoi "żywy" (i kosztuje), dopóki go nie zatrzymasz albo dopóki
  nie zadziała auto-termination (patrz niżej).
- **Job cluster** -- klaster tworzony AUTOMATYCZNIE na potrzeby jednego zaplanowanego
  zadania (Job, sekcja 6) i niszczony zaraz po jego zakończeniu -- tańszy, bo nie stoi
  bezczynnie między uruchomieniami.
- **SQL Warehouse** -- compute zoptymalizowane pod zapytania SQL (np. z dashboardów albo
  narzędzi BI), z osobnym silnikiem strojenia wydajności zapytań.
- **Serverless** -- najprostsza opcja (i domyślna w Free Edition): w ogóle nie
  konfigurujesz maszyn, typów instancji ani skalowania -- platforma zarządza tym
  automatycznie w tle.

**Konfiguracja klastra** (gdy NIE korzystasz z serverless) obejmuje: **Databricks
Runtime** (wersja Sparka + zestaw prekonfigurowanych bibliotek -- istnieje osobny
*ML Runtime* z gotowym TensorFlow/PyTorch/MLlib), typ instancji (ile CPU/RAM na maszynę),
**autoscaling** (klaster sam dokłada i usuwa executory w zależności od obciążenia, dokładnie
to, o czym mówiliśmy koncepcyjnie w Module 2 architektury klastra), **spot/preemptible
instances** (tańsze, przerywalne maszyny dla zadań tolerujących restart) oraz
**auto-termination** (klaster sam się wyłącza po N minutach bezczynności, żeby nie płacić
za stojącą bezczynnie maszynę).

**Photon** to przyspieszony, natywny silnik wykonawczy Databricks (napisany w C++, nie w
JVM) -- opcjonalne przyspieszenie dla zapytań SQL i DataFrame, transparentne dla Twojego
kodu (nie zmieniasz ani linijki PySparka, żeby z niego skorzystać).

> ⚠️ **Klastry kosztują (nawet bezczynne)**
>
> All-purpose cluster nalicza opłaty, dopóki działa -- NIEZALEŻNIE od tego, czy w danej chwili coś liczy. Auto-termination (np. "zatrzymaj po 30 minutach bezczynności") to jedno z pierwszych ustawień, które warto sprawdzić, zanim zostawisz notebook na noc.

> 🧱 **Cluster policies**
>
> W środowisku firmowym administratorzy zwykle definiują *cluster policies* -- gotowe szablony konfiguracji (np. "maks. 4 węzły, tylko spot instances, auto-termination po 20 minutach"), z których zwykli użytkownicy wybierają, zamiast konfigurować klaster od zera. To sposób na kontrolę kosztów i spójność w całej organizacji.

<a id="sec4"></a>
## 4. `dbutils` — scyzoryk szwajcarski notebooków

`dbutils` to specjalny obiekt dostępny automatycznie w KAŻDYM notebooku Databricks
(podobnie jak `spark`) -- zestaw narzędzi do zadań, których nie da się wygodnie zrobić
samym PySparkiem. Cztery najważniejsze podmoduły:

**`dbutils.fs`** -- operacje na systemie plików (DBFS -- Databricks File System -- oraz
zamontowanych magazynach w chmurze):

```
dbutils.fs.ls("/mnt/dane")           # lista plikow, jak `ls` w terminalu
dbutils.fs.cp("zrodlo", "cel")       # kopiowanie
dbutils.fs.mv("stara_sciezka", "nowa_sciezka")
dbutils.fs.rm("sciezka", recurse=True)
```

**`dbutils.notebook`** -- uruchamianie i łańcuchowanie notebooków z poziomu kodu (a nie
tylko z Jobs UI):

```
# uruchamia inny notebook i CZEKA na wynik (w przeciwienstwie do %run,
# ktory laczy przestrzenie nazw -- .run() traktuje go jak "podprogram")
wynik = dbutils.notebook.run("./przetworz_dane", timeout_seconds=600, arguments={"data": "2024-01-01"})

# na KONCU notebooka, ktory jest wywolywany w ten sposob:
dbutils.notebook.exit("sukces: przetworzono 3000 wierszy")
```

To fundament budowania **potoków** (pipelines) z wielu mniejszych notebooków, zamiast
jednego olbrzymiego pliku.

**`dbutils.secrets`** -- bezpieczny dostęp do sekretów (hasła, klucze API) bez wpisywania
ich wprost w kodzie notebooka:

```
haslo = dbutils.secrets.get(scope="moj-zespol", key="haslo_bazy")
```

Sekrety wpisuje się osobno (przez CLI albo API) do *secret scope* -- nigdy nie widać ich
wartości w samym notebooku ani w historii wersji, co jest kluczowe dla bezpieczeństwa
(dokładnie ten problem, o którym uczyliśmy się przy `.env` i zmiennych środowiskowych w
kursie Pythona).

**`dbutils.widgets`** -- omawiamy osobno w sekcji 5, bo to jeden z najczęściej używanych
podmodułów w praktyce.

> ⚡ **`dbutils.fs` a `%fs`**
>
> Te same operacje na plikach można też wywołać jako magic command w komórce: `%fs ls /mnt/dane` zamiast `dbutils.fs.ls("/mnt/dane")`. Wynik `%fs` wyświetla się od razu jako czytelna tabela -- wygodne przy szybkiej eksploracji, `dbutils.fs.ls()` wygodniejsze, gdy chcesz przetworzyć wynik dalej w kodzie.

<a id="sec5"></a>
## 5. Widgety — parametryzacja notebooków

**Widgety** (`dbutils.widgets`) zamieniają notebook z pojedynczego skryptu w
**parametryzowalny szablon** -- ten sam notebook można uruchomić dla różnych wartości
(np. różnych dat, środowisk, segmentów klientów) bez zmiany ani linijki kodu.

```
# na poczatku notebooka -- definiujemy widget (raz; kolejne uruchomienia go nie nadpisuja)
dbutils.widgets.text("data_raportu", "2024-01-01", "Data raportu")
dbutils.widgets.dropdown("srodowisko", "dev", ["dev", "test", "produkcja"], "Srodowisko")
dbutils.widgets.multiselect("segmenty", "Premium", ["Nowy", "Standardowy", "Premium"], "Segmenty")

# w dowolnym miejscu ponizej -- odczytujemy wartosc
data_raportu = dbutils.widgets.get("data_raportu")
srodowisko = dbutils.widgets.get("srodowisko")

zamowienia_z_dnia = zamowienia.filter(zamowienia.data_zamowienia == data_raportu)
```

W interfejsie Databricks te widgety pojawiają się jako pola tekstowe / listy rozwijane
na górze notebooka -- ktoś, kto nie zna Sparka, może uruchomić notebook z innymi
parametrami, po prostu zmieniając wartość w polu, zamiast edytować kod.

Typy widgetów: `text` (dowolny tekst), `dropdown` (jedna wartość z listy), `combobox`
(lista + możliwość wpisania własnej wartości), `multiselect` (wiele wartości naraz).

> 🧱 **Widgety + Jobs = parametryzowane zadania produkcyjne**
>
> Prawdziwa moc widgetów ujawnia się w połączeniu z Jobs (sekcja 6): definiując zadanie, podajesz KONKRETNE wartości parametrów (np. `data_raportu = {{job.start_time}}` -- specjalna zmienna z datą uruchomienia zadania), więc ten sam notebook, uruchamiany codziennie o 6:00, automatycznie przetwarza dane za odpowiedni dzień -- bez ręcznej edycji.

<a id="sec6"></a>
## 6. Jobs i Workflows — planowanie zadań

**Jobs** (w nowszym UI: **Workflows**) to mechanizm Databricks do uruchamiania notebooków
i skryptów AUTOMATYCZNIE, bez ręcznego klikania "Run" -- fundament produkcyjnych
potoków danych.

**Zadanie wieloetapowe (multi-task job).** Pojedyncze zadanie może składać się z wielu
**tasków** ułożonych w graf zależności (DAG) -- np. Task A ("wczytaj i wyczyść dane")
musi zakończyć się sukcesem, zanim wystartuje Task B ("policz agregaty"), a Task B musi
się zakończyć, zanim wystartują RÓWNOLEGLE Task C ("trenuj model MLlib") i Task D
("zapisz raport SQL"). To dokładnie ten sam pomysł co orkiestracja wielu notebooków
przez `dbutils.notebook.run()` (sekcja 4), ale zarządzany wizualnie, z historią uruchomień,
retry i alertami wbudowanymi w platformę.

**Triggery** -- kiedy zadanie ma wystartować:

- **Harmonogram (schedule)** -- wyrażenie w stylu cron, np. "codziennie o 6:00".
- **Przybycie pliku (file arrival)** -- zadanie startuje automatycznie, gdy w
  monitorowanej lokalizacji pojawi się nowy plik (typowe dla pipeline'ów event-driven).
- **Ciągłe (continuous)** -- zadanie działa bez przerwy, restartując się natychmiast po
  zakończeniu (typowe dla przetwarzania strumieniowego).
- **Ręczne / na żądanie** -- uruchamiane przez człowieka albo wywołanie API/CLI.

**Job cluster** (wspomniany w sekcji 3) to domyślny typ compute dla zadań: tworzony na
start zadania, niszczony po jego zakończeniu -- taniej niż trzymanie all-purpose clustera
włączonego cały czas w oczekiwaniu na codzienne uruchomienie.

**Retry i alerty.** Każdy task można skonfigurować tak, żeby przy błędzie automatycznie
spróbował ponownie (np. 3 razy, z przerwą 5 minut) oraz żeby wysłał powiadomienie
(e-mail, Slack) w razie sukcesu, porażki albo przekroczenia czasu.

> 💡 **Databricks Asset Bundles (DAB)**
>
> Dla zespołów, które chcą trzymać konfigurację jobów, klastrów i notebooków jako kod (Infrastructure as Code) i wdrażać ją przez CI/CD -- podobnie jak Terraform dla infrastruktury chmurowej -- Databricks oferuje **Asset Bundles**: pliki YAML opisujące cały projekt, wdrażane komendą `databricks bundle deploy`. To temat wykraczający poza ten kurs, ale warto wiedzieć, że istnieje, gdy zespół dojrzewa od pojedynczych notebooków do pełnego procesu wdrożeniowego.

<a id="sec7"></a>
## 7. Praktyczny spacerek po Databricks Free Edition

Czas przełożyć powyższą teorię na praktykę we własnym koncie **Databricks Free Edition**
(założonym w Module 1). Poniższe kroki wykonaj samodzielnie w przeglądarce:

1. **Zaimportuj notebook.** W Workspace kliknij prawym przyciskiem na swój folder →
   *Import* → wgraj dowolny plik `.ipynb` z tego kursu (np. `Spark_Modul3...ipynb`).
   Databricks otworzy go dokładnie tak, jak wygląda tutaj.
2. **Utwórz i podłącz compute.** W sekcji *Compute* utwórz nowy klaster (albo skorzystaj
   z domyślnego serverless) i podłącz go do zaimportowanego notebooka (przycisk *Connect*
   w prawym górnym rogu notebooka).
3. **Uruchom notebook.** *Run All* uruchamia wszystkie komórki po kolei -- powinieneś
   zobaczyć identyczne wyniki jak podczas pracy lokalnej w tym kursie.
4. **Dodaj widget.** W nowej, pustej komórce na górze notebooka wpisz
   `dbutils.widgets.text("segment_filtr", "Premium", "Segment")`, uruchom komórkę --
   u góry notebooka pojawi się pole tekstowe. Zmień jego wartość i w kolejnej komórce
   odczytaj ją: `dbutils.widgets.get("segment_filtr")`.
5. **Utwórz prosty Job.** W sekcji *Workflows* utwórz nowe zadanie z jednym taskiem
   wskazującym na Twój notebook, ustaw harmonogram (np. "raz dziennie o 8:00") i
   zapisz -- NIE musisz czekać na uruchomienie, sam widok konfiguracji wystarczy, żeby
   zobaczyć, jak działa DAG tasków i wybór triggera.

> ⚡ **Nie masz jeszcze konta?**
>
> Wróć do Modułu 1, sekcja 6 -- tam opisaliśmy krok po kroku, jak założyć darmowe konto Databricks Free Edition (wystarczy adres e-mail, bez własnego konta w chmurze).

<a id="sec8"></a>
## 8. Podsumowanie i ćwiczenia

W tym module poznaliśmy platformę Databricks (nie nową składnię Sparka):
- anatomię workspace'u: Workspace, Repos, Catalog, Compute, Workflows,
- notebooki wielojęzyczne (`%python`/`%sql`/`%scala`/`%md`), historię wersji i `%run`,
- typy compute (all-purpose, job cluster, SQL Warehouse, serverless) i ich konfigurację
  (Runtime, autoscaling, spot instances, auto-termination, Photon),
- `dbutils` -- `fs`, `notebook`, `secrets` i (szerzej) `widgets`,
- parametryzację notebooków widgetami,
- Jobs/Workflows: multi-task joby, triggery, retry i alerty,
- praktyczne pierwsze kroki we własnym koncie Databricks Free Edition.

Poniższe ćwiczenia wykonaj w swoim koncie Databricks Free Edition -- nie da się ich
sprawdzić lokalnie, bo wymagają prawdziwego `dbutils` i Jobs UI.

### 📝 Ćwiczenie 1: Zaimportuj i uruchom

W Databricks Free Edition zaimportuj notebook z Modułu 5 (Spark SQL) i uruchom go w całości (*Run All*) na podłączonym compute. Sprawdź, czy wynik ostatniej komórki zgadza się z tym, co widziałeś lokalnie.

<details>
<summary>Pokaż rozwiązanie</summary>

```text
1. Workspace -> Twoj folder -> Import -> wybierz plik .ipynb Modulu 5.
2. Compute -> utworz/wybierz klaster (albo uzyj domyslnego serverless).
3. W otwartym notebooku: przycisk "Connect" w prawym gornym rogu -> wybierz compute.
4. Run All. Wynik ostatniej komorki (np. wypisane kolumny z spark.catalog.listColumns)
   powinien byc identyczny jak podczas lokalnego wykonania w tym kursie.
```

</details>

### 📝 Ćwiczenie 2: Dodaj dwa widgety

W dowolnym zaimportowanym notebooku dodaj widget typu `dropdown` z listą miast (`Warszawa`, `Krakow`, `Gdansk`, `Wroclaw`, `Poznan`) oraz widget typu `text` z domyślną wartością segmentu (`Premium`). Odczytaj obie wartości i użyj ich do przefiltrowania DataFrame'u `klienci`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
dbutils.widgets.dropdown("miasto", "Warszawa", ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"], "Miasto")
dbutils.widgets.text("segment", "Premium", "Segment")

miasto = dbutils.widgets.get("miasto")
segment = dbutils.widgets.get("segment")

klienci.filter((klienci.miasto == miasto) & (klienci.segment == segment)).show()
```

</details>

### 📝 Ćwiczenie 3: Zaplanuj zadanie

W sekcji Workflows utwórz nowe zadanie (Job) z jednym taskiem wskazującym na notebook z ćwiczenia 2. Ustaw harmonogram na "codziennie o 6:00" oraz włącz powiadomienie e-mail w razie porażki zadania.

<details>
<summary>Pokaż rozwiązanie</summary>

```text
1. Workflows -> Create Job.
2. Task: wybierz "Notebook" jako typ, wskaz notebook z Cwiczenia 2, wybierz compute
   (job cluster albo istniejacy).
3. Schedule -> Add trigger -> Scheduled -> cron: "0 6 * * *" (codziennie o 6:00).
4. Notifications -> dodaj swoj e-mail przy zdarzeniu "On failure".
5. Save. (Uruchomienie na zadanie nie jest wymagane do wykonania cwiczenia.)
```

</details>

> 🔥 **Wyzwanie: mini-potok z dwóch notebooków**
>
> Utwórz DRUGI notebook, który przyjmuje widget `min_wartosc` (typu `text`, wartość domyślna `"100"`) i wypisuje liczbę zamówień o wartości większej niż podana (potrzebna kolumna `wartosc` z Modułu 12 -- policz ją ponownie joinem zamówień z pozycjami). W PIERWSZYM notebooku wywołaj drugi przez `dbutils.notebook.run("./nazwa_drugiego", 300, {"min_wartosc": "500"})` i wypisz zwrócony wynik.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
# Notebook 2 ("prog_wartosci"):
dbutils.widgets.text("min_wartosc", "100")
prog = float(dbutils.widgets.get("min_wartosc"))

from pyspark.sql import functions as F
zamowienia_z_wartoscia = (
    pozycje_zamowien.groupBy("zamowienie_id")
    .agg(F.sum(F.col("ilosc") * F.col("cena_jednostkowa") * (1 - F.col("rabat"))).alias("wartosc"))
)
liczba = zamowienia_z_wartoscia.filter(F.col("wartosc") > prog).count()
dbutils.notebook.exit(f"Zamowien powyzej {prog}: {liczba}")

# Notebook 1 (wywolujacy):
wynik = dbutils.notebook.run("./prog_wartosci", 300, {"min_wartosc": "500"})
print(wynik)
```

dbutils.notebook.run() zwraca dokladnie to, co przekazane do .exit() w drugim notebooku -- to najprostszy sposob na przekazywanie wynikow miedzy notebookami bez wspoldzielonych zmiennych globalnych.

</details>

## Co dalej?

W **Module 15**, ostatnim module kursu, połączymy WSZYSTKO — od wczytania surowych danych,
przez czyszczenie i transformacje DataFrame API, zapytania Spark SQL, aż po wytrenowanie
modelu MLlib i zapis wyników — w jeden kompletny, wykonywalny potok danych od początku do
końca.